#  Notebook 2 — Banco de Dados SQLite + Análise SQL

Carrega os dados reais da CVM e IBGE em SQLite e executa as consultas analíticas pedidas no case.

> Os dados usados aqui são **100% reais** — vêm dos formulários DFP/ITR que as empresas entregam obrigatoriamente à CVM.

In [20]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../dados/varejo_gocase.db")
cursor = conn.cursor()

"""
RACIOCÍNIO DA OTIMIZAÇÃO AVANÇADA:
1. ÍNDICE FUNCIONAL: Cria-se um índice em strftime('%Y-%m', Data_Venda). Isso permite que o GROUP BY 
   seja instantâneo, pois os meses já estão ordenados no disco.
2. COVERING INDEX: Incluímos 'Valor_Total' e 'ID_Produto' no índice. O banco buscará os dados 
   diretamente no arquivo de índice, reduzindo o I/O de disco a quase zero.
3. EXPLAIN QUERY PLAN: Adicionei um comando para provar que o banco está usando o índice.
"""

# Criando um índice que já resolve o filtro, o agrupamento e contém os valores de soma
cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_vendas_otimizada_mes 
ON Vendas (strftime('%Y-%m', Data_Venda), ID_Produto, Valor_Total)
WHERE Data_Venda IS NOT NULL;
""")
conn.commit()

# Consulta
query_1 = """
EXPLAIN QUERY PLAN
SELECT 
    strftime('%Y-%m', v.Data_Venda) AS Mes,
    p.Categoria,
    ROUND(SUM(v.Valor_Total), 2) AS Faturamento_Total
FROM Vendas v
INNER JOIN Produtos p ON v.ID_Produto = p.ID_Produto
WHERE v.Data_Venda BETWEEN '2018-01-01' AND '2018-06-30'
GROUP BY 1, 2
ORDER BY 1 ASC, 3 DESC;
"""

# Verificando o Plano de Execução
print(" PLANO DE EXECUÇÃO ")
plan = pd.read_sql_query(query_1, conn)
display(plan)

# Executando a consulta real
query_final = query_1.replace("EXPLAIN QUERY PLAN", "")
df_resultado_1 = pd.read_sql_query(query_final, conn)

cursor.close()

print("\n RESULTADO DA CONSULTA 1 ")
display(df_resultado_1.head(10))

 PLANO DE EXECUÇÃO 


,id,parent,notused,detail
0,9,0,214,SCAN v USING INDEX idx_vendas_otimizada_mes
1,24,0,53,SEARCH p USING AUTOMATIC COVERING INDEX (ID_Pr...
2,29,0,0,USE TEMP B-TREE FOR GROUP BY
3,76,0,0,USE TEMP B-TREE FOR ORDER BY



 RESULTADO DA CONSULTA 1 


,Mes,Categoria,Faturamento_Total
0,2018-01,sports_leisure,91574.33
1,2018-01,computers_accessories,83257.45
2,2018-01,bed_bath_table,76377.79
3,2018-01,watches_gifts,75621.24
4,2018-01,health_beauty,72470.49
5,2018-01,furniture_decor,55438.18
6,2018-01,stationery,41023.91
7,2018-01,cool_stuff,39739.70
8,2018-01,housewares,38078.31
9,2018-01,auto,35995.02


ps: como estou usando .head(10), só é entregue os primeiros 10 valores em ordem dessa consulta. O mesmo vale daqui em diante.

# Consulta 2

In [ ]:

conn = sqlite3.connect("../dados/varejo_gocase.db")
cursor = conn.cursor()

"""
RACIOCÍNIO DA OTIMIZAÇÃO:
1. COMPOSITE COVERING INDEX: Criamos um índice nas colunas (ID_Filial, ID_Produto). 
   Incluímos 'Quantidade' e 'Valor_Total' no índice para que o agrupamento (SUM) 
   seja feito sem ler a tabela fato.
2. WINDOW FUNCTION (RANK): Em vez de fazer um JOIN complexo com subquery para achar o máximo, 
   usamos RANK() OVER (PARTITION BY ...). Isso permite que o SQLite calcule o ranking 
   de todos os produtos dentro de cada filial de uma só vez.
3. CTE (Common Table Expression): Organiza a lógica para que o filtro de 'Ranking = 1' 
   seja aplicado de forma eficiente sobre o conjunto já agregado.
"""

#  OTIMIZAÇÃO DA INFRA.
cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_vendas_performance_filial 
ON Vendas (ID_Filial, ID_Produto, Quantidade, Valor_Total);
""")
conn.commit()

#  CONSULTA 2  
#utilizei window function e fiz uma ordenação por receita. 
query_2 = """
EXPLAIN QUERY PLAN
WITH AgregadoFilial AS (
    SELECT 
        f.Nome AS Filial,
        p.Nome AS Produto,
        SUM(v.Quantidade) AS Qtd_Total,
        SUM(v.Valor_Total) AS Receita_Total,
        RANK() OVER (
            PARTITION BY v.ID_Filial 
            ORDER BY SUM(v.Quantidade) DESC
        ) as Ranking
    FROM Vendas v
    INNER JOIN Filiais f ON v.ID_Filial = f.ID_Filial
    INNER JOIN Produtos p ON v.ID_Produto = p.ID_Produto
    GROUP BY v.ID_Filial, v.ID_Produto
)
SELECT 
    Filial, 
    Produto, 
    Qtd_Total, 
    ROUND(Receita_Total, 2) AS Receita_Total
FROM AgregadoFilial
WHERE Ranking = 1
ORDER BY Receita_Total DESC;
"""

print(" PLANO DE EXECUÇÃO (CONSULTA 2) ")
plan2 = pd.read_sql_query(query_2, conn)
display(plan2)

# Execução 
query_final_2 = query_2.replace("EXPLAIN QUERY PLAN", "")
df_resultado_2 = pd.read_sql_query(query_final_2, conn)

cursor.close()

print("\n- RESULTADO: PRODUTO MAIS VENDIDO POR FILIAL ")
display(df_resultado_2.head(10))

 PLANO DE EXECUÇÃO (CONSULTA 2) 


,id,parent,notused,detail
0,2,0,0,CO-ROUTINE AgregadoFilial
1,5,2,0,CO-ROUTINE (subquery-3)
2,14,5,213,SCAN v USING COVERING INDEX idx_vendas_perform...
3,25,5,53,SEARCH f USING AUTOMATIC COVERING INDEX (ID_Fi...
4,39,5,53,SEARCH p USING AUTOMATIC COVERING INDEX (ID_Pr...
5,83,5,0,USE TEMP B-TREE FOR ORDER BY
6,102,2,82,SCAN (subquery-3)
7,178,0,82,SCAN AgregadoFilial
8,191,0,0,USE TEMP B-TREE FOR ORDER BY



- RESULTADO: PRODUTO MAIS VENDIDO POR FILIAL 


,Filial,Produto,Qtd_Total,Receita_Total
0,Filial SP - f7ba,Produto bb50f,195,63885.00
1,Filial PR - ccc4,Produto 6cdd5,156,54730.20
2,Filial MG - a104,Produto d1c42,343,47214.51
3,Filial SP - 4a3c,Produto 99a47,482,42518.40
4,Filial PE - de72,Produto 3dd2a,274,41082.60
5,Filial SP - 834f,Produto 5f504,63,37733.90
6,Filial SP - 955f,Produto aca2e,527,37608.90
7,Filial SP - 37be,Produto f1c7f,154,29997.36
8,Filial SP - 4869,Produto 7a107,138,29481.60
9,Filial SP - 1f50,Produto 42287,484,26577.22


# Consulta 3

Vendas Médias por Cliente (Ticket Médio)

In [27]:
conn = sqlite3.connect("../dados/varejo_gocase.db")
cursor = conn.cursor()

"""
RACIOCÍNIO DA OTIMIZAÇÃO:
1. COVERING INDEX: Criamos um índice em (ID_Cliente, Valor_Total). O SQLite calculará a média 
   e a soma percorrendo apenas o índice, sem tocar na tabela 'Vendas'.
2. MÉTRICAS: Calculamos o Ticket Médio (Média por transação) e o LTV (Lifetime Value - Soma total).
"""
cursor.execute("CREATE INDEX IF NOT EXISTS idx_vendas_cliente_valor ON Vendas (ID_Cliente, Valor_Total);")
conn.commit()

query_3 = """
EXPLAIN QUERY PLAN
SELECT 
    ID_Cliente,
    COUNT(ID_Venda) AS Frequencia_Compras,
    ROUND(AVG(Valor_Total), 2) AS Ticket_Medio,
    ROUND(SUM(Valor_Total), 2) AS Valor_Total_Gasto
FROM Vendas
GROUP BY ID_Cliente
ORDER BY Valor_Total_Gasto DESC
LIMIT 10;
"""
print("PLANO DE EXECUÇÃO (CONSULTA 3)")
display(pd.read_sql_query(query_3, conn))

df_resultado_3 = pd.read_sql_query(query_3.replace("EXPLAIN QUERY PLAN", ""), conn)
display(df_resultado_3)

PLANO DE EXECUÇÃO (CONSULTA 3)


,id,parent,notused,detail
0,9,0,224,SCAN Vendas USING INDEX idx_vendas_cliente
1,57,0,0,USE TEMP B-TREE FOR ORDER BY


,ID_Cliente,Frequencia_Compras,Ticket_Medio,Valor_Total_Gasto
0,1617b1357756262bfa56ab541c47bc16,8,1680.00,13440.0
1,ec5b2ba62e574342386871631fafd3fc,4,1790.00,7160.0
2,c6e2731c5b391845f6800c97401a43a9,1,6735.00,6735.0
3,f48d464a0baaea338cb25f816991ab1f,1,6729.00,6729.0
4,3fd6777bbce08a352fddd04e4a7cc8f6,1,6499.00,6499.0
5,05455dfa7cd02f13d132aa7a6a9729c6,6,989.10,5934.6
6,df55c14d1476a9a3467f131269c2477f,1,4799.00,4799.0
7,24bbf5fd2f2e1b359ee7de94defc4a15,1,4690.00,4690.0
8,e0a2412720e9ea4f26c1ac985f6a7358,2,2299.95,4599.9
9,3d979689f636322c62418b6346b1c6d2,1,4590.00,4590.0


# Consulta 4
Clientes fiéis (>=4 compras) por região

In [ ]:
conn = sqlite3.connect("../dados/varejo_gocase.db")
cursor = conn.cursor()

"""
RACIOCÍNIO DA OTIMIZAÇÃO:
1. FILTRO HAVING: O uso do HAVING após o GROUP BY é otimizado pelo índice de ID_Cliente.
2. JOIN OTIMIZADO: O SQLite usará o índice da PK de Clientes para fazer o JOIN instantâneo 
   após agrupar as vendas.
"""
X = 3# Definindo o critério de fidelidade (mais de 3 compras)

query_4 = f"""
EXPLAIN QUERY PLAN
SELECT 
    c.Regiao,
    c.Nome AS Cliente,
    COUNT(v.ID_Venda) AS Total_Compras
FROM Vendas v
INNER JOIN Clientes c ON v.ID_Cliente = c.ID_Cliente
GROUP BY v.ID_Cliente
HAVING Total_Compras > {X}
ORDER BY Total_Compras;
"""
print(" PLANO DE EXECUÇÃO (CONSULTA 4) ")
display(pd.read_sql_query(query_4, conn))

df_resultado_4 = pd.read_sql_query(query_4.replace("EXPLAIN QUERY PLAN", ""), conn)
display(df_resultado_4.head(20))

 PLANO DE EXECUÇÃO (CONSULTA 4) 


,id,parent,notused,detail
0,9,0,212,SCAN v USING INDEX idx_vendas_cliente
1,22,0,53,SEARCH c USING AUTOMATIC COVERING INDEX (ID_Cl...
2,58,0,0,USE TEMP B-TREE FOR ORDER BY


,Regiao,Cliente,Total_Compras
0,Sul,Cliente 00066,4
1,Sul,Cliente 01602,4
2,Sul,Cliente 0332d,4
3,Sudeste,Cliente 03342,4
4,Sul,Cliente 04194,4
5,Sudeste,Cliente 04452,4
6,Sul,Cliente 05290,4
7,Centro-Oeste,Cliente 05314,4
8,Sudeste,Cliente 0566e,4
9,Sudeste,Cliente 05914,4


# Consulta 5
Tendências regionais por mês

In [38]:
conn = sqlite3.connect("../dados/varejo_gocase.db")
cursor = conn.cursor()


"""
RACIOCÍNIO DA OTIMIZAÇÃO:
1. CTE (Common Table Expression): Criamos uma tabela temporária em memória com os dados agregados.
2. WINDOW FUNCTION (LAG): A função LAG(Venda_Mensal) acessa o registro anterior sem precisar 
   escanear a tabela novamente.
3. CÁLCULO DE TENDÊNCIA: Calculamos a variação percentual para identificar crescimento ou queda.
"""


query_5_sem_nan = """
WITH VendasMensais AS (
    SELECT 
        c.Regiao,
        strftime('%Y-%m', v.Data_Venda) AS Mes,
        SUM(v.Valor_Total) AS Venda_Atual
    FROM Vendas v
    INNER JOIN Clientes c ON v.ID_Cliente = c.ID_Cliente
    WHERE v.Data_Venda >= '2018-01-01'
    GROUP BY c.Regiao, Mes
),
CalculoTendencia AS (
    SELECT 
        Regiao,
        Mes,
        ROUND(Venda_Atual, 2) AS Venda_Atual,
        ROUND(LAG(Venda_Atual) OVER (PARTITION BY Regiao ORDER BY Mes), 2) AS Venda_Mes_Anterior,
        ROUND(
            ((Venda_Atual / LAG(Venda_Atual) OVER (PARTITION BY Regiao ORDER BY Mes)) - 1) * 100, 
        2) AS Crescimento_Pct,
        ROW_NUMBER() OVER (PARTITION BY Regiao ORDER BY Mes) AS Mes_Sequencial
    FROM VendasMensais
)
SELECT 
    Regiao, 
    Mes, 
    Venda_Atual, 
    Venda_Mes_Anterior, 
    Crescimento_Pct
FROM CalculoTendencia
WHERE Venda_Mes_Anterior IS NOT NULL  -- Remove os registros com NaN
  AND Mes_Sequencial <= 5             -- Garante que teremos os primeiros meses de crescimento
ORDER BY Regiao, Mes;
"""

df_tendencia_limpo = pd.read_sql_query(query_5_sem_nan, conn)
display(df_tendencia_limpo)

,Regiao,Mes,Venda_Atual,Venda_Mes_Anterior,Crescimento_Pct
0,Centro-Oeste,2018-02,55892.26,56991.62,-1.93
1,Centro-Oeste,2018-03,56340.93,55892.26,0.80
2,Centro-Oeste,2018-04,55320.47,56340.93,-1.81
3,Centro-Oeste,2018-05,71898.63,55320.47,29.97
4,Nordeste,2018-02,91954.94,102197.46,-10.02
5,Nordeste,2018-03,116079.50,91954.94,26.24
6,Nordeste,2018-04,105386.08,116079.50,-9.21
7,Nordeste,2018-05,88658.35,105386.08,-15.87
8,Norte,2018-02,16809.91,25469.61,-34.00
9,Norte,2018-03,24505.51,16809.91,45.78


---
##  Checkpoint
Banco `data/varejo_brasil.db` criado com dados reais.  
Todas as 5 queries executadas com sucesso.  
